# 01 · XDF → CSV

生の LSL 記録をセッション単位のワイド CSV に変換する.

| | |
|---|---|
| **入力** | `data/raw/<subject>/*.xdf` |
| **出力** | `data/csv/<subject>/<session>.csv` |

各 `.xdf` には同時記録された 3 ストリーム（EEG=`EmotivDataStream-EEG`, 表面筋電=`EMG_Stream`, モーションキャプチャ=`OptiTrack_BiomechIDs`）が入っている. 共通の LSL タイムスタンプで統合し, 列名はストリーム種別で接頭（`EEG_`, `EMG_`, `Markers_`）する.

> 実験データは公開していない（参加者のプライバシー保護）. このノートブックを 実行するには自前の記録を `data/raw/` 以下に配置する.

In [ ]:
import sys
from pathlib import Path

# notebooks/ から実行したときに motion_intent パッケージを import 可能にする
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_intent import config

In [ ]:
from motion_intent.io_xdf import summarize_xdf, xdf_to_dataframe, export_session_csv

## Inspect one recording

In [ ]:
xdf_files = sorted(config.RAW_DIR.rglob('*.xdf'))
assert xdf_files, f'no .xdf files under {config.RAW_DIR} - place your recordings there first'
print(f'{len(xdf_files)} xdf files under {config.RAW_DIR}')

summarize_xdf(xdf_files[0])

## Convert every session

`export_session_csv` は `<out_dir>/<拡張子違いの同名>.csv` を書き出す. 被験者フォルダ名は各 `.xdf` の親ディレクトリ名から取る.

In [ ]:
for xdf_path in xdf_files:
    subject = xdf_path.parent.name
    out_dir = config.CSV_DIR / subject
    csv_path = export_session_csv(xdf_path, out_dir)
    print(csv_path.relative_to(config.DATA_DIR))

## Quick look at the merged frame

In [ ]:
if xdf_files:
    df = xdf_to_dataframe(xdf_files[0])
    print(df.shape)
    display(df.filter(regex='^(t_sec|EEG_Cz|EMG_|Markers_).*').head())